# 00 - Raw image conversion and dataset preparation

This notebook documents the one-time data-preparation workflow used to convert ANSA `.tif` image stacks into PyTorch tensor files (`.pt`), assign tensors to train/validation/test splits, and reduce full 180-frame tensors to the 7-frame model input used for training.

The training notebooks do not depend on the original local raw-image directory. For reproducible GitHub/Docker use, the preferred workflow is:

1. Download the prepared `.pt` tensor files and split manifests from OSF.
2. Rebuild the expected train/validation/test folder structure from the manifests.
3. Train and validate the clinical or logarithmic models using the rebuilt folders.

This notebook is included for transparency and for regenerating the processed tensors from `.tif` stacks when those image files are available.


## Notebook role in the repository

This notebook should be treated as a data-provenance notebook rather than a required training notebook. It explains how `.tif` stacks were converted into tensors, how split manifests can be generated, and how full time-series tensors were reduced to the 7-frame inputs used by the ResNet models.

The Docker training workflow should instead use OSF-hosted `.pt` tensors plus manifest-based dataset rebuilding, because the raw image stacks are large and are not distributed with the GitHub repository.


In [ ]:
from pathlib import Path
import random
import shutil
import sys

import pandas as pd
import torch
import torch.nn.functional as F
from skimage.io import imread

# Allow imports from the local src/ package when this notebook is run from either
# the repository root or the notebooks/ folder.
NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ansa.data_utils import (
    DEFAULT_SELECTED_FRAME_INDICES,
    reduce_dataset_to_7_frame_tensors,
)


## Configuration

Update the paths below for your local machine or Docker mount.

Expected raw input structure:

```text
RAW_TIF_ROOT/
├── 0/
│   ├── image_001.tif
│   └── ...
├── 100/
│   ├── image_001.tif
│   └── ...
└── ...
```

The folder name is interpreted as the copy-number label. Output tensors are saved as `{copy_number}_{replicate}.pt`.

The workflow creates two split-folder datasets:

- `FULL_FRAME_SPLIT_ROOT`: split/class folders containing full 180-frame tensors.
- `REDUCED_7_FRAME_SPLIT_ROOT`: split/class folders containing the final 7-frame tensors used by model training.


In [ ]:
# Local or Docker-adjustable paths
RAW_TIF_ROOT = Path("/workspace/data/raw_tifs")
FULL_FRAME_TENSOR_ROOT = Path("/workspace/data/interpolated_torch_tensors")
FULL_FRAME_SPLIT_ROOT = Path("/workspace/data/logarithmic_split_180_frame")
REDUCED_7_FRAME_SPLIT_ROOT = Path("/workspace/data/logarithmic_split_7_frame")
MANIFEST_ROOT = Path("/workspace/manifests")

TARGET_SIZE = (500, 500)
EXPECTED_FRAMES = 180
BASELINE_FRAME_COUNT = 20
SELECTED_FRAME_INDICES = DEFAULT_SELECTED_FRAME_INDICES
SEED = 53

# Same split proportions used in the original exploratory notebook.
SPLIT_FRACTIONS = {
    "Training": 0.60,
    "Testing": 0.20,
    "Validation": 0.20,
}

for path in [FULL_FRAME_TENSOR_ROOT, FULL_FRAME_SPLIT_ROOT, REDUCED_7_FRAME_SPLIT_ROOT, MANIFEST_ROOT]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
LOG_CLASS_ORDER = [
    "0.undetectable",
    "1.low",
    "2.medium",
    "3.high",
    "4.very high",
]


def logarithmic_class(copy_number: int) -> str:
    """Map a numeric copy number to the logarithmic class label."""
    if copy_number < 100:
        return "0.undetectable"
    if copy_number <= 1_000:
        return "1.low"
    if copy_number <= 10_000:
        return "2.medium"
    if copy_number <= 100_000:
        return "3.high"
    return "4.very high"


def discover_tif_stacks(raw_root: Path) -> pd.DataFrame:
    """Return one row per TIFF stack found under copy-number folders."""
    records = []
    for label_dir in sorted(raw_root.iterdir()):
        if not label_dir.is_dir():
            continue
        try:
            copy_number = int(label_dir.name)
        except ValueError:
            print(f"Skipping non-numeric label folder: {label_dir.name}")
            continue

        tif_files = sorted(
            list(label_dir.glob("*.tif")) + list(label_dir.glob("*.tiff"))
        )
        for replicate, tif_path in enumerate(tif_files, start=1):
            tensor_name = f"{copy_number}_{replicate}.pt"
            records.append({
                "copy_number": copy_number,
                "replicate": replicate,
                "class_label": logarithmic_class(copy_number),
                "source_tif": str(tif_path),
                "tensor_file": tensor_name,
            })
    return pd.DataFrame.from_records(records)


def convert_tif_to_tensor(tif_path: Path, output_path: Path, target_size=(500, 500)) -> torch.Size:
    """Load one TIFF stack, convert to [1, T, H, W], resize, and save as .pt."""
    image_stack = imread(tif_path)
    tensor = torch.as_tensor(image_stack, dtype=torch.float32)

    if tensor.ndim != 3:
        raise ValueError(f"Expected [T, H, W] stack, got shape {tuple(tensor.shape)} for {tif_path}")

    if tensor.shape[0] != EXPECTED_FRAMES:
        raise ValueError(f"Expected {EXPECTED_FRAMES} frames, got {tensor.shape[0]} for {tif_path}")

    # [T, H, W] -> [1, T, H, W]
    tensor = tensor.unsqueeze(0)

    # Match the original notebook's interpolation behavior.
    resized = F.interpolate(
        tensor,
        size=target_size,
        mode="bicubic",
        align_corners=True,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(resized, output_path)
    return resized.shape


def split_within_class(df: pd.DataFrame, seed: int = 53) -> pd.DataFrame:
    """Assign train/test/validation splits within each class for more stable class balance."""
    rng = random.Random(seed)
    split_rows = []

    for class_label, group in df.groupby("class_label", sort=True):
        rows = group.to_dict("records")
        rng.shuffle(rows)

        n = len(rows)
        n_train = int(n * SPLIT_FRACTIONS["Training"])
        n_test = int(n * SPLIT_FRACTIONS["Testing"])

        for idx, row in enumerate(rows):
            if idx < n_train:
                split = "Training"
            elif idx < n_train + n_test:
                split = "Testing"
            else:
                split = "Validation"
            row["split"] = split
            split_rows.append(row)

    return pd.DataFrame(split_rows).sort_values(["split", "class_label", "tensor_file"]).reset_index(drop=True)


def copy_split_tensors(manifest: pd.DataFrame, tensor_root: Path, split_root: Path) -> None:
    """Copy tensor files into split/class folders expected by PyTorch ImageFolder-style loaders."""
    for _, row in manifest.iterrows():
        source = tensor_root / row["tensor_file"]
        destination = split_root / row["split"] / row["class_label"] / row["tensor_file"]
        destination.parent.mkdir(parents=True, exist_ok=True)

        if not source.exists():
            raise FileNotFoundError(f"Missing tensor file: {source}")
        shutil.copy2(source, destination)


## Step 1 - Build the raw-image index

This table is the bridge between the original `.tif` files and the processed `.pt` tensor filenames. Save it so that the conversion can be audited later.


In [ ]:
samples = discover_tif_stacks(RAW_TIF_ROOT)
print(f"Discovered {len(samples)} TIFF stacks")
display(samples.head())

sample_manifest_path = MANIFEST_ROOT / "sample_manifest.csv"
samples.to_csv(sample_manifest_path, index=False)
print(f"Saved sample manifest: {sample_manifest_path}")


## Step 2 - Convert `.tif` stacks to full-frame `.pt` tensors

Each raw stack is saved as a float32 tensor with shape `[1, 180, 500, 500]`. This is the full temporal representation. A later step reduces these files to the 7-frame model input.


In [ ]:
converted_shapes = []

for _, row in samples.iterrows():
    tif_path = Path(row["source_tif"])
    output_path = FULL_FRAME_TENSOR_ROOT / row["tensor_file"]
    shape = convert_tif_to_tensor(tif_path, output_path, target_size=TARGET_SIZE)
    converted_shapes.append(tuple(shape))

samples["tensor_shape"] = converted_shapes
print(samples["tensor_shape"].value_counts())
print(f"Saved full-frame tensors to: {FULL_FRAME_TENSOR_ROOT}")


## Step 3 - Create train/test/validation splits

The original exploratory notebook used a 60/20/20 split with random seed 53. Here, the same proportions and seed are retained, but splitting is performed within each logarithmic class so that the class distribution is more stable across splits.

For exact historical reproduction of a previously published split, use the saved split manifest rather than regenerating it.


In [ ]:
log_manifest = split_within_class(samples, seed=SEED)

split_manifest_path = MANIFEST_ROOT / "logarithmic_split_manifest.csv"
log_manifest.to_csv(split_manifest_path, index=False)

print(f"Saved split manifest: {split_manifest_path}")
display(pd.crosstab(log_manifest["class_label"], log_manifest["split"]))


## Step 4 - Rebuild full-frame split/class folders

This creates split/class folders for the full 180-frame tensors. These folders are an intermediate dataset used before reduction to the final 7-frame model inputs.

```text
logarithmic_split_180_frame/
├── Training/
│   ├── 0.undetectable/
│   ├── 1.low/
│   ├── 2.medium/
│   ├── 3.high/
│   └── 4.very high/
├── Testing/
└── Validation/
```

The next step reduces this folder structure to the 7-frame version used by model training.


In [ ]:
copy_split_tensors(log_manifest, FULL_FRAME_TENSOR_ROOT, FULL_FRAME_SPLIT_ROOT)
print(f"Copied full-frame tensors into split folders under: {FULL_FRAME_SPLIT_ROOT}")

for split in ["Training", "Testing", "Validation"]:
    print(f"{split}")
    for class_label in LOG_CLASS_ORDER:
        class_dir = FULL_FRAME_SPLIT_ROOT / split / class_label
        count = len(list(class_dir.glob("*.pt"))) if class_dir.exists() else 0
        print(f"  {class_label}: {count}")


## Step 5 - Reduce full-frame tensors to 7-frame model inputs

The ResNet models use 7 input frames per sample. The first frame is the average of the first 20 frames, followed by six selected frames from later in the time series:

```text
[mean(frames 0-19), frame 69, frame 89, frame 109, frame 129, frame 149, frame 179]
```

The implementation lives in `src/ansa/data_utils.py` as `reduce_dataset_to_7_frame_tensors()` so that the preprocessing logic can be reused by notebooks, scripts, and tests without copy-pasting code.


In [ ]:
for split in ["Training", "Testing", "Validation"]:
    written_files = reduce_dataset_to_7_frame_tensors(
        root_dir=FULL_FRAME_SPLIT_ROOT / split,
        save_dir=REDUCED_7_FRAME_SPLIT_ROOT / split,
        class_names=LOG_CLASS_ORDER,
        selected_frame_indices=SELECTED_FRAME_INDICES,
        baseline_frame_count=BASELINE_FRAME_COUNT,
        target_size=TARGET_SIZE,
        overwrite=True,
    )
    print(f"{split}: wrote {len(written_files)} reduced tensors")

print(f"Saved 7-frame tensors to: {REDUCED_7_FRAME_SPLIT_ROOT}")


## Quality-control notes

Before using these tensors for training, verify:

- all full-frame tensors have shape `[1, 180, 500, 500]` before 7-frame reduction;
- all reduced tensors have shape `[1, 7, 500, 500]` after reduction;
- all expected copy-number classes are represented;
- train, validation, and test splits are generated once and then preserved by manifest;
- the clinical and logarithmic models use separate manifests if their labels or splits differ;
- OSF-hosted tensor files are byte-identical to the files referenced by the manifests.


In [ ]:
def summarize_tensor_folder(tensor_root: Path, recursive: bool = False) -> pd.DataFrame:
    """Summarize tensor shape, dtype, and value range for `.pt` files."""
    pattern = "**/*.pt" if recursive else "*.pt"
    records = []
    for pt_path in sorted(tensor_root.glob(pattern)):
        tensor = torch.load(pt_path, map_location="cpu")
        records.append({
            "tensor_file": pt_path.name,
            "relative_path": str(pt_path.relative_to(tensor_root)),
            "shape": tuple(tensor.shape),
            "dtype": str(tensor.dtype),
            "min": float(tensor.min()),
            "max": float(tensor.max()),
        })
    return pd.DataFrame(records)

full_frame_qc = summarize_tensor_folder(FULL_FRAME_TENSOR_ROOT)
print("Full-frame tensor shapes:")
print(full_frame_qc["shape"].value_counts())
display(full_frame_qc.head())

reduced_qc = summarize_tensor_folder(REDUCED_7_FRAME_SPLIT_ROOT, recursive=True)
print("Reduced 7-frame tensor shapes:")
print(reduced_qc["shape"].value_counts())
display(reduced_qc.head())
